# Federated MedMNIST 3D - Orchestrator

This notebook launches a **real** federated learning experiment on distributed Flower infrastructure.

```
┌──────────────────────────────────────────┐
│  HOST_SuperLink  (SuperLink + Orchestrator) │
│  flower-superlink                        │
│    :9092  ← SuperNodes connect here      │
│    :9093  ← flwr run connects here       │
└──────────────┬───────────────────────────┘
               │ gRPC
   ┌───────────┴────────────┐
   │                        │
┌──┴──────────────┐  ┌──────┴──────────────┐
│ HOST_SuperNode  │  │ HOST_SuperNode       │
│ SuperNode 0     │  │ SuperNode 1          │
│ local port 9094 │  │ local port 9096      │
└─────────────────┘  └─────────────────────┘
```

## Before running

1. **HOST_SuperLink** - start the SuperLink: `bash start_superlink.sh`
2. **HOST_SuperNode terminal 1** - `bash start_supernode.sh <SUPERLINK_IP>`
3. **HOST_SuperNode terminal 2** - `bash start_supernode.sh <SUPERLINK_IP> 9096`
4. Set `SUPERLINK_IP` in the cell below and run all cells.

> ⚠️ SuperNodes must have `torch` and `medmnist` already installed,
> or be started with `--allow-runtime-dependency-installation`.

## 0 - Imports & configuration

In [1]:
import importlib
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
import workshop_helpers

importlib.reload(workshop_helpers)
from workshop_helpers import bin_dir, preflight, print_port_status, run_stream, update_superlink_address

ctx = preflight("fl_medmnist3d")
flwr_bin_prefix = bin_dir(ctx)

SUPERLINK_IP = "127.0.0.1"
SUPERLINK_FLEET_PORT = 9092
SUPERLINK_CONTROL_PORT = 9093

Python executable : /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/.venv-workshop/bin/python
Python version    : 3.13.5
Notebook cwd      : /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/real_fed
Flower app        : /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/real_fed/fl_medmnist3d
flwr binary       : /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/.venv-workshop/bin/flwr


flwr            : 1.30.0 @ /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/.venv-workshop/lib/python3.13/site-packages/flwr
numpy           : 2.4.6 @ /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/.venv-workshop/lib/python3.13/site-packages/numpy


The helper functions above keep notebook setup out of the way. The remaining cells show the Flower commands in the same shape students would run from a terminal.

## 1 - Check SuperLink connectivity

In [2]:
superlink_ready = print_port_status(
    SUPERLINK_IP,
    {"Fleet API": SUPERLINK_FLEET_PORT, "Control API": SUPERLINK_CONTROL_PORT},
)

Fleet API    127.0.0.1:9092  -> not reachable
Control API  127.0.0.1:9093  -> not reachable


## Distributed Run: Deployment Runtime

This follows Flower's Deployment Runtime flow:

1. Start one `flower-superlink` process.
2. Start two `flower-supernode` processes and pass `--node-config` so each SuperNode gets a distinct partition.
3. Add/use a named SuperLink connection, `local-deployment`, pointing to the SuperLink Control API.
4. Submit the MedMNIST 3D app with `flwr run ... local-deployment --stream`.

For a real deployment, run the SuperLink and SuperNodes on their own machines and use TLS instead of `--insecure`.

In [3]:
SUPERLINK_IP = "127.0.0.1"
SUPERLINK_FLEET_PORT = 9092
SUPERLINK_CONTROL_PORT = 9093
DEPLOYMENT_CONNECTION = "local-deployment"

superlink_ready = print_port_status(
    SUPERLINK_IP,
    {"Fleet API": SUPERLINK_FLEET_PORT, "Control API": SUPERLINK_CONTROL_PORT},
)

command = r"""
flwr run fl_medmnist3d local-deployment \
--run-config num-server-rounds=3 \
--run-config min-clients=2 \
--run-config num-partitions=2 \
--run-config local-epochs=1 \
--run-config learning-rate=0.001 \
--stream
""".strip()

if superlink_ready:
    update_superlink_address(f"{SUPERLINK_IP}:{SUPERLINK_CONTROL_PORT}", name=DEPLOYMENT_CONNECTION)
    rc = run_stream(flwr_bin_prefix + command, cwd=ctx.root)
    if rc != 0:
        raise RuntimeError(f"Distributed MedMNIST run failed with exit code {rc}")
else:
    print("SuperLink is not reachable; skipping distributed run.")
    print("Command to run after SuperLink and SuperNodes are started:")
    print(command)

Fleet API    127.0.0.1:9092  -> not reachable
Control API  127.0.0.1:9093  -> not reachable
SuperLink is not reachable; skipping distributed run.
Command to run after SuperLink and SuperNodes are started:
flwr run fl_medmnist3d local-deployment \
--run-config num-server-rounds=3 \
--run-config min-clients=2 \
--run-config num-partitions=2 \
--run-config local-epochs=1 \
--run-config learning-rate=0.001 \
--stream


## 3 - Local test (optional)

Start SuperLink and 2 SuperNodes **on the same machine** to test without HOST_SuperNode.
Useful to verify the code works before switching to the real network.

In [4]:
command = r"""
flwr run fl_medmnist3d local \
--run-config num-server-rounds=3 \
--run-config min-clients=2 \
--run-config num-partitions=2 \
--run-config local-epochs=1 \
--run-config learning-rate=0.001 \
--federation-config num-supernodes=2 \
--stream
""".strip()

rc = run_stream(flwr_bin_prefix + command, cwd=ctx.root)
if rc != 0:
    raise RuntimeError(f"Local MedMNIST simulation failed with exit code {rc}")

$ /Users/pfoley/Documents/Codex/2026-05-21/i-have-a-colleague-who-is/FL_Demos/.venv-workshop/bin/flwr run fl_medmnist3d local \
  --run-config num-server-rounds=3 \
  --run-config min-clients=2 \
  --run-config num-partitions=2 \
  --run-config local-epochs=1 \
  --run-config learning-rate=0.001 \
  --federation-config num-supernodes=2 \
  --stream


🎊 Successfully started run 11837208591355836404


INFO :      Starting logstream for run_id `11837208591355836404`


INFO :      Starting Flower Simulation


INFO :      Starting FedAvg strategy:


...

  2: {'accuracy': '7.2881e-01', 'eval_loss': '5.9004e-01'},


  3: {'accuracy': '7.2881e-01', 'eval_loss': '5.8472e-01'}}


ServerApp-side Evaluate Metrics:


{}


INFO :      
